In [0]:
# ============================================================
# Silver Layer - Customer Dimension Enrichment
# ============================================================

from pyspark.sql.functions import *

# Read Bronze Table
customer_df = spark.table("workspace.pei_assignment.customer_raw")

# Customer Transformations
customer_enriched = (
    customer_df
        .dropDuplicates(["customer_id"])
        .withColumn("customer_name", initcap(trim(col("customer_name"))))
        .withColumn("email", lower(trim(col("email"))))
        .withColumn("phone", regexp_replace(col("phone"), "[^0-9]", ""))
        .withColumn(
            "is_valid_email",
            col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
        )
        .withColumn(
            "is_valid_phone",
            length(col("phone")) >= 10
        )
        .withColumn("load_timestamp", current_timestamp())
)

print("="*60)
print("Customer Silver Validation")
print("="*60)
print(f"Total Records : {customer_enriched.count()}")

display(customer_enriched)

customer_enriched.printSchema()

(
    customer_enriched.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.pei_assignment.customer_enriched")
)

print("customer_enriched created successfully")

Customer Silver Validation
Total Records : 793


customer_id,customer_name,email,phone,address,segment,country,city,state,postal_code,region,is_valid_email,is_valid_phone,load_timestamp
PW-19240,Pierre Wener,bettysullivan808@gmail.com,42158009029815,"001 Jones Ridges Suite 338 Johnsonfort, FL 95462",Consumer,United States,Louisville,Colorado,80027,West,true,true,2026-07-21T21:05:12.654Z
GH-14410,Gary567 Hansen,austindyer948@gmail.com,0015424150246314,"00347 Murphy Unions Ashleyton, IA 29814",Home Office,United States,Chicago,Illinois,60653,Central,true,true,2026-07-21T21:05:12.654Z
KL-16555,Kelly Lampkin,clarencehughes280@gmail.com,7185624866,"007 Adams Lane Suite 176 East Amyberg, IN 34581",Corporate,United States,Colorado Springs,Colorado,80906,West,true,true,2026-07-21T21:05:12.654Z
AH-10075,Ad. ..am Hart,angelabryant256@gmail.com,26510155691098,"01454 Christopher Turnpike North Ryanstad, MI 36226",Corporate,United States,Columbus,Ohio,43229,East,true,true,2026-07-21T21:05:12.654Z
PF-19165,Philip Fox,kristinereynolds576@gmail.com,00147364521419154,"0158 Harris Ways Suite 085 East Laceyside, SD 35649",Consumer,United States,San Diego,California,92105,West,true,true,2026-07-21T21:05:12.654Z
SC-20680,Steve Carroll,jasoncontreras178@gmail.com,56364748305318,"01630 Tammy Prairie North Daniel, KS 26404",Home Office,United States,Seattle,Washington,98105,West,true,true,2026-07-21T21:05:12.654Z
JR-15700,Jocasta Rupert,johncombs689@gmail.com,6181,"019 Emily Corner Apt. 810 Ryantown, SC 37010",Consumer,United States,Jacksonville,Florida,32216,South,true,false,2026-07-21T21:05:12.654Z
AB-10105,Adrian Barton,daviddavis980@gmail.com,0674358553692,"021 Katherine Mall Jameston, DC 24685",Consumer,United States,Phoenix,Arizona,85023,West,true,true,2026-07-21T21:05:12.654Z
PT-19090,Pete@#$ Takahito,mikaylaarnold666@gmail.com,7866386820,"0236 Lane Squares Port Samantha, ME 15670",Consumer,United States,San Antonio,Texas,78207,Central,true,true,2026-07-21T21:05:12.654Z
SG-20605,Speros Goranitis,brianjoyce110@gmail.com,3528465094,"02401 Angela Loop Apt. 678 Port John, ME 43448",Consumer,United States,Lafayette,Indiana,47905,Central,true,true,2026-07-21T21:05:12.654Z


root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- address: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- is_valid_email: boolean (nullable = true)
 |-- is_valid_phone: boolean (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)

customer_enriched created successfully


In [0]:
# ============================================================
# Silver Layer - Product Dimension Enrichment
# ============================================================

from pyspark.sql.functions import *

# Read Bronze Table
products_df = spark.table("workspace.pei_assignment.products_raw")

# Product Transformations
product_enriched = (
    (
    products_df
        .dropDuplicates(["product_id"])
        .withColumn("product_name", initcap(trim(col("product_name"))))
        .withColumn("category", initcap(trim(col("category"))))
        .withColumn("sub_category", initcap(trim(col("sub_category"))))
        .withColumn("state", initcap(trim(col("state"))))
        .withColumn("load_timestamp", current_timestamp())
)

        # Remove commas and currency symbols
        .withColumn(
            "price_per_product",
            regexp_replace(col("price_per_product"), "[$,]", "")
        )

        # Cast only valid numeric values
        .withColumn(
            "price_per_product",
            when(
                col("price_per_product").rlike(r"^[0-9]+(\.[0-9]+)?$"),
                col("price_per_product").cast("double")
            ).otherwise(None)
        )

        .withColumn("load_timestamp", current_timestamp())
)

# Validation
print("=" * 60)
print("Product Silver Validation")
print("=" * 60)
print(f"Total Records : {product_enriched.count()}")
print(f"Total Columns : {len(product_enriched.columns)}")

display(product_enriched)

product_enriched.printSchema()

# Write Silver Table
(
    product_enriched.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.pei_assignment.product_enriched")
)

print("product_enriched created successfully")

# Verify
spark.sql("""
SELECT COUNT(*) AS total_records
FROM workspace.pei_assignment.product_enriched
""").show()

Product Silver Validation
Total Records : 1818
Total Columns : 7


product_id,category,sub_category,product_name,state,price_per_product,load_timestamp
FUR-CH-10002961,Furniture,Chairs,"Leather Task Chair, Black",New York,81.882,2026-07-21T21:05:17.583Z
TEC-AC-10004659,Technology,Accessories,Imation secure+ Hardware Encrypted Usb 2.0 flash Drive; 16gb,Oklahoma,72.99,2026-07-21T21:05:17.583Z
OFF-BI-10002824,Office Supplies,Binders,Recycled Easel Ring Binders,Colorado,4.25,2026-07-21T21:05:17.583Z
OFF-PA-10003349,Office Supplies,Paper,Xerox 1957,Florida,5.184,2026-07-21T21:05:17.583Z
TEC-AC-10003023,Technology,Accessories,Logitech G105 Gaming Keyboard,Ohio,47.496,2026-07-21T21:05:17.583Z
OFF-BI-10004233,Office Supplies,Binders,"""gbc Pre-punched Binding Paper, Plastic, White, 8-1/2"""" X 11""""""",New Jersey,15.99,2026-07-21T21:05:17.583Z
OFF-PA-10004470,Office Supplies,Paper,"""adams Write N' Stick Phone Message Book, 11"""" X 5 1/4""""","200 Messages""",null,2026-07-21T21:05:17.583Z
FUR-FU-10001196,Furniture,Furnishings,Dax Cubicle Frames - 8x10,Indiana,5.77,2026-07-21T21:05:17.583Z
OFF-ST-10000585,Office Supplies,Storage,Economy Rollaway Files,Kentucky,165.2,2026-07-21T21:05:17.583Z
OFF-ST-10003996,Office Supplies,Storage,"Letter/legal File Tote With Clear Snap-on Lid, Black Granite",Washington,16.06,2026-07-21T21:05:17.583Z


root
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- price_per_product: double (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)

product_enriched created successfully
+-------------+
|total_records|
+-------------+
|         1818|
+-------------+



In [0]:
products_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- price_per_product: string (nullable = true)

